# Part 1: Word2Vec
---

## 1. Imports & Reproducibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import random

import pickle
from datasets import load_dataset
import torch as torch

np.random.seed(5)
random.seed(5)


---
## 2. Dataset Loading

Dataset Loading and token collection is done to split upon.

In [ ]:
dataset = load_dataset('lhoestq/conll2003')

#list of sentences, each a list of tokens
all_sentences = []
for split in ['train', 'validation', 'test']:
    for example in dataset[split]:
        all_sentences.append(example['tokens'])

print(f'sentences: {len(all_sentences):,}')
print(f'tokens: {sum(len(s) for s in all_sentences):,}')


---
## 3. Preprocessing

### Design decisions

- **Lowercasing**
- **Punctuation removal**

We then build:
- `vocab`: `word to index` mapping (sorted alphabetically for reproducibility)
- `idx2word`: `index to word` reverse mapping
- `word_counts`: raw frequency of each token (needed for negative-sampling distribution)

In [ ]:
import string

punct = set(string.punctuation)

def is_punct_only(token):
    return len(token) > 0 and all(ch in punct for ch in token) #only a punctuation point

def preprocess(sentences):
    #lowercase & drop punctuation
    processed= []
    #list of sentences each a list of tokens
    for sent in sentences:
        clean = [tok.lower() for tok in sent if not is_punct_only(tok)] #list of non punct tokens in a sentence
        if clean:                      #skip empty sents
            processed.append(clean)

    word_counts= Counter(tok for sent in processed for tok in sent)

    vocab_words= sorted(word_counts.keys()) #all unique words sorted - to reprod
    vocab= {w: i for i, w in enumerate(vocab_words)}    #word : i
    idx2word= {i: w for w, i in vocab.items()}  # i: word

    return processed, vocab, idx2word, word_counts

processed_sentences, vocab, idx2word, word_counts = preprocess(all_sentences)

v_size = len(vocab)
print(v_size)

---
## 4. Unigram distribution

Positive samples are observed from text and positive pairs are to be generated below. However, for negative sampling, since pairs are created using (word, -ve context), its more appropriate to use unigram distribution for this case

$$
P_{\text{neg}}(w) = \frac{f(w)^\alpha}{\sum_{u \in V} f(u)^\alpha}
$$

Where:

- $f(w)$ = raw frequency of word \(w\) in the corpus  
- $\alpha$ = 0.75; reduces the dominance of very frequent words  
- The denominator normalizes the distribution so that $(\sum_{w \in V} P_{\text{neg}}(w) = 1)$


In [ ]:
def build_noise_distribution(vocab, word_counts, power=0.75):

    noise_dist = np.zeros(len(vocab))
    for word, idx in vocab.items():
        noise_dist[idx]= word_counts[word] ** power
    noise_dist /= noise_dist.sum()   #normalise - probability
    return noise_dist


noise_dist = build_noise_distribution(vocab, word_counts)

print(f'Noise distr shape: {noise_dist.shape}')

## 5. Skip-Gram Training Pair Generation
Steps:
- A window of size 2 here is slid over each sentence.  
- For every position a positive pair is generated (word, context)
- All pairs are stored as a NumPy integer array of shape (N, 2) for fast mini-batch indexing.

In [ ]:
# Hyperparameters
WINDOW_SIZE= 2
EMBED_DIM= 100    #embedding dimensionality
NUM_NEG=10       #k negative samples per positive pair
LEARNING_RATE = 0.025
EPOCHS= 10      #passes over all training pairs
BATCH_SIZE= 512    #mini-batch size
MIN_DELTA = 1e-4
P= 2 

def generate_skipgram_pairs(sentences, vocab, window_size=WINDOW_SIZE):

    pairs = []
    for sent in sentences:  #for each sentence
        indices = [vocab[w] for w in sent]  # sentence of indices
        n = len(indices)
        for i in range(n):
            lo = max(0, i - window_size)
            hi = min(n, i + window_size + 1)
            for j in range(lo, hi):
                if j != i: #paie word with each context in bounds
                    pairs.append((indices[i], indices[j]))
    return np.array(pairs, dtype=np.int32)

pairs = generate_skipgram_pairs(processed_sentences, vocab)


---
## 6. Word2Vec Model:

### Two embedding matrices W_in: target word, W_out: context

---

### Loss function

$$
L_{CE} = -\left[\log \sigma(c_{\text{pos}} \cdot w) + \sum_{i=1}^{k} \log \sigma(-c_{\text{neg}_i} \cdot w)\right]
$$

Since $\sigma(-x) = 1 - \sigma(x)$:

$$
L_{CE} = -\left[\log \sigma(c_{\text{pos}} \cdot w) + \sum_{i=1}^{k} \log (1 - \sigma(c_{\text{neg}_i} \cdot w))\right]
$$

---

### Gradients

$$\frac{\partial L}{\partial c_{\text{pos}}} = [\sigma(c_{\text{pos}} \cdot w) - 1]\, w$$

$$\frac{\partial L}{\partial c_{\text{neg}}} = [\sigma(c_{\text{neg}} \cdot w)]\, w$$

$$\frac{\partial L}{\partial w} = [\sigma(c_{\text{pos}} \cdot w) - 1]\,c_{\text{pos}} + \sum_{i=1}^{k}[\sigma(c_{\text{neg}_i} \cdot w)]\,c_{\text{neg}_i}$$

---

### SGD update equations

$$c_{\text{pos}}^{t+1} = c_{\text{pos}}^{t} - \eta\,[\sigma(c_{\text{pos}}^{t} \cdot w^{t}) - 1]\,w^{t}$$

$$c_{\text{neg}}^{t+1} = c_{\text{neg}}^{t} - \eta\,[\sigma(c_{\text{neg}}^{t} \cdot w^{t})]\,w^{t}$$

$$w^{t+1} = w^{t} - \eta\left[[\sigma(c_{\text{pos}} \cdot w^{t}) - 1]\,c_{\text{pos}} + \sum_{i=1}^{k}[\sigma(c_{\text{neg}_i} \cdot w^{t})]\,c_{\text{neg}_i}\right]$$

In [ ]:
class SkipGramNegativeSampling:

    def __init__(self, vocab_size, embed_dim, num_neg, noise_dist):
        self.vocab_size = vocab_size
        self.embed_dim= embed_dim
        self.num_neg= num_neg
        self.noise_dist= noise_dist

        #random starting vectors, centered around zero - divided by dimension of embedding no exploding grad
        self.W_in  = ((np.random.rand(vocab_size, embed_dim) - 0.5) / embed_dim).astype(np.float32)
        self.W_out = ((np.random.rand(vocab_size, embed_dim) - 0.5) / embed_dim).astype(np.float32)

    @staticmethod
    def sigmoid(x):
        x = np.clip(x, -500, 500)
        return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)

    def sample_negatives(self, batch_size):
        return np.random.choice(self.vocab_size,size=(batch_size, self.num_neg),p=self.noise_dist)      #for w target words, k neg each matrix w x k


    def train_batch(self, target_indices, context_indices, lr):
        """
            L = -[log sigma(c_pos . w)  +  sum_k log sigma(-c_neg_k . w)]

        Gradients:
            dL/d_c_pos   = [sigma(c_pos . w) - 1] * w
            dL/d_c_neg_k = [sigma(c_neg_k . w)]   * w
            dL/d_w       = [sigma(c_pos . w) - 1] * c_pos + sum_k [sigma(c_neg_k . w)] * c_neg_k

        Updates:
            c_pos_new = c_pos - lr * dL/d_c_pos
            c_neg_new = c_neg - lr * dL/d_c_neg
            w_new = w - lr * dL/d_w
        """
        B= len(target_indices)

        w= self.W_in[target_indices]         #target vectors
        c_pos= self.W_out[context_indices]       #positive context

        neg_idx= self.sample_negatives(B)        #negative indices
        c_neg= self.W_out[neg_idx]             #negative context

        s_pos = np.einsum('bd,bd->b',   c_pos, w)     #c_pos . w
        s_neg = np.einsum('bkd,bd->bk', c_neg, w)     #c_neg_k . w

        sig_pos = self.sigmoid(s_pos)             #sigma(c_pos . w)
        sig_neg = self.sigmoid(s_neg)             #sigma(c_neg . w)

        eps= 1e-9    #prevent log0
        loss_pos = -np.log(sig_pos + eps)
        loss_neg= -np.log(1.0 - sig_neg + eps)
        avg_loss= (loss_pos.sum() + loss_neg.sum()) / B    #average over batch

        #gadients
        err_pos = sig_pos - 1.0     #sigma(c_pos.w) - 1
        err_neg = sig_neg   #sigma(c_neg.w)

        # dL/d_c_pos= err_pos * w -- elementwise B, * B,D
        grad_c_pos = err_pos[:, None] * w

        # dL/d_c_neg = err_neg * w  -- B,K, / B, ,D
        grad_c_neg = err_neg[:, :, None] * w[:, None, :]

        # dL/d_w = err_pos*c_pos + sum_k(err_neg*c_neg)
        grad_w = (err_pos[:, None] * c_pos + np.einsum('bk,bkd->bd', err_neg, c_neg))

        #update context embeddings for positive words -- handle indices
        np.add.at(self.W_out, context_indices, -lr * grad_c_pos)

        # update context embeddings for negative words
        np.add.at(self.W_out, neg_idx.reshape(-1), (-lr * grad_c_neg).reshape(-1, self.embed_dim))

        # update target embeddings
        np.add.at(self.W_in, target_indices, -lr * grad_w)

        return avg_loss


    def get_embeddings(self):
         return self.W_in + self.W_out


# init
model = SkipGramNegativeSampling(
    vocab_size=v_size,
    embed_dim=EMBED_DIM,
    num_neg=NUM_NEG,
    noise_dist=noise_dist
)

---
## 7. Training

Mini-batch SGD with **linear learning-rate decay**:

$$\eta_t = \max\!\left(\eta_0 \cdot \left(1 - \frac{t}{T_{\text{total}}}\right),\; \eta_{\min}\right)$$



In [ ]:

MIN_DELTA = 1e-4
PATIENCE  = 2      

def train_word2vec(model, pairs, epochs=EPOCHS, batch_size=BATCH_SIZE,lr_init=LEARNING_RATE, lr_min=1e-4,min_delta=MIN_DELTA, patience=PATIENCE):
    N= len(pairs)
    batches_per_epoch = N // batch_size
    total_batches= epochs * batches_per_epoch
    loss_history= []    # for plotting
    epoch_loss_history= []    # one value per epoch, used for early stopping
    batch_num = 0
    LOG_EVERY= max(1, total_batches // 300)    # ~300 log points

    # Early stopping state
    best_loss= float('inf')
    no_improve_count  = 0

    print(f'Training  |  pairs={N:,}  epochs={epochs}  batch={batch_size}')
    print(f'          |  total_batches={total_batches:,}')
    print(f'          |  early_stopping: patience={patience}, min_delta={min_delta}')
    print()

    for epoch in range(1, epochs + 1):
        # Shuffle pairs at start of each epoch
        perm= np.random.permutation(N)
        pairs = pairs[perm]

        epoch_loss= 0.0
        epoch_batches= 0

        for start in range(0, N - batch_size + 1, batch_size):
            batch= pairs[start : start + batch_size]
            targets= batch[:, 0]   #center word indices
            contexts = batch[:, 1]   #positive context indices

            #Linear LR decay
            progress= batch_num / max(total_batches, 1)
            lr= max(lr_init * (1.0 - progress), lr_min)

            loss= model.train_batch(targets, contexts, lr)

            epoch_loss+= loss
            epoch_batches+= 1
            batch_num+= 1

            if batch_num % LOG_EVERY == 0:
                loss_history.append(loss)

        avg = epoch_loss / max(epoch_batches, 1)
        epoch_loss_history.append(avg)

        print(f'  Epoch {epoch}/{epochs}  avg_loss={avg:.4f}  '
              f'lr={lr:.5f} ')

        if best_loss - avg > min_delta:
            best_loss= avg
            no_improve_count = 0   # reset patience counter on improvement
        else:
            no_improve_count += 1
            print(f'  [Early Stopping] No significant improvement '
                  f'({no_improve_count}/{patience})')
            if no_improve_count >= patience:
                print(f'  [Early Stopping] Triggered after epoch {epoch}. '
                      f'Best avg loss: {best_loss:.4f}')
                break

    print('\nTraining complete.')
    return loss_history, epoch_loss_history


loss_history, epoch_loss_history = train_word2vec(model, pairs)


In [ ]:
# Plot 1: batch-level loss curve
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(loss_history, linewidth=1.0, color='steelblue', alpha=0.85)
axes[0].set_title('Batch-level Loss (sampled)', fontsize=12)
axes[0].set_xlabel('Logging Step')
axes[0].set_ylabel('L_CE')
axes[0].grid(True, alpha=0.3)

# Plot 2: epoch-level loss early stopping 
axes[1].plot(range(1, len(epoch_loss_history) + 1), epoch_loss_history,
             marker='o', linewidth=1.5, color='darkorange')
axes[1].set_title('Epoch-level Avg Loss (early stopping criterion)', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Avg L_CE')
axes[1].set_xticks(range(1, len(epoch_loss_history) + 1))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('word2vec_loss.png', dpi=150)
plt.show()
print('Loss curves saved -> word2vec_loss.png')


---
## 8. Save Embeddings
Saving for future use

In [ ]:
embeddings = model.get_embeddings()    # shape (VOCAB_SIZE, EMBED_DIM)

np.save('embeddings.npy', embeddings)
print(f'Saved embeddings.npy   shape={embeddings.shape}  dtype={embeddings.dtype}')

with open('vocab.pkl', 'wb') as f:
    pickle.dump({'vocab': vocab, 'idx2word': idx2word}, f)
print('Saved vocab.pkl')


embeddings.npy:
The optimized $W_{in}$ weight matrix.
Each row is a 100D coordinate representing a word's unique position in the semantic manifold.

vocab.pkl: The translation layer.

*  vocab: Maps strings to matrix indices (Word $\rightarrow$ Row ID)
*   idx2word: Maps indices back to strings (Row ID $\rightarrow$ Word)


---
# Part 2: Using Embeddings in Word Analogy Function
---

## First: Visualitization of embeddings.npy and vocab.pkl

In [ ]:
# loading embeddings.npy
embeddings = np.load('embeddings.npy')

# seeing the sahpe
print(f"Shape of embeddings: {embeddings.shape}")

# looking at the vector for first word in matrix
print("First word's vector sample:")
print(embeddings[0][:10]) # shows first 10 dim

# loading vocab.pkl
with open('vocab.pkl', 'rb') as f:
    data = pickle.load(f)

vocab = data['vocab']
idx2word = data['idx2word']

# length of vocab
print(f"Vocabulary size: {len(vocab)}")

# few examples of words& their indices
print("Sample of vocabulary:")
for word in list(vocab.keys())[:10]:
    print(f"Word: '{word}' -> Index: {vocab[word]}")

In [ ]:
import re

clean_words = [
    word.lower()
    for word in vocab.keys()
    if re.fullmatch(r"[a-zA-Z']+", word)
]

## Second: Word Analogy Function

The goal is to find the missing word $B^*$ that completes the relationship established by the pair $(A, B)$. This is achieved through Vector Arithmetic:$$V_{B^*} \approx V_B - V_A + V_{A^*}$$Relationship Extraction ($V_B - V_A$): This calculates the "transformation vector." It captures the specific logical shift such as moving from present to past tense, or from one month to the next.Relationship Application ($+ V_{A^*}$):By adding this transformation vector to the new starting word $A^*$, we arrive at a coordinate in the 100D space where the answer $B^*$ should logically reside.

**Cosine Similarity** between 2 vectors u and v
$\mathbf{u}$ and $\mathbf{v}$ is defined as:$$\text{similarity} = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$

In [ ]:
import numpy as np

def word_analogies_fn(test_list, top_k=5):
    """
    Solves analogies and prints the Top 5 results to show semantic clustering.
    """
    print(f"{'Analogy Task (A:B :: A*:?)':<40} | {'Top 5 Predictions (Word : Score)'}")
    print("=" * 120)

    # normalising embeddings for cleaner results & not favouring words with high frequencies
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normed_embeds = embeddings / (norms + 1e-9)

    #identifying & skipping any analogy test cases where the input words are missing from the trained dict, preventing runtime errors
    for a, b, a_star in test_list:
        if not all(w in vocab for w in [a, b, a_star]):
            missing = [w for w in [a, b, a_star] if w not in vocab]
            print(f"{f'{a}:{b} :: {a_star}':<40} | Missing: {', '.join(missing)}")
            continue

        # vector arithmetic
        v_a = normed_embeds[vocab[a]]
        v_b = normed_embeds[vocab[b]]
        v_as = normed_embeds[vocab[a_star]]

        target = v_b - v_a + v_as
        # scales target to unit length,to ensure search is based strictly on cosine similarity
        target = target / (np.linalg.norm(target) + 1e-9)

        #  cosine similarity
        sims = np.dot(normed_embeds, target) #calculation
        best_idx = np.argsort(sims)[::-1]    # ranking

        # collecting top 5 words possible for target
        results = []
        for idx in best_idx:
            word = idx2word[idx]
            if word not in [a, b, a_star]:
                results.append(f"{word} ({sims[idx]:.2f})")
            if len(results) >= top_k:
                break

        print(f"{f'{a}:{b} :: {a_star}':<40} | {', '.join(results)}")

# test cases using words that are well_represented in the dataset we are using
tests = [
    # past tense
    ('take', 'took', 'give'),       #  should be: gave
    ('say', 'said', 'go'),          # went / saw

    # sequential logic
    ('january', 'february', 'march'), # april/may

    # 3. direct opposities
    ('west', 'east', 'north'),       # south
    ('new', 'old', 'good'),          # bad / better

    # 4. quantity
    ('one', 'two', 'first')          # second
]

word_analogies_fn(tests)

## Third: Word Analogy Patterns in Vector Space

**1. Past Tense Transformation**  
take took, give gave (0.91)  
The model captures verb tense changes as consistent directional shifts in embedding space.

**2. Domain Bias in News Language**  
say said, go reported (0.64)  
Shows contextual drift where verbs can take reporting-style meanings in news corpora.

**3. Sequential Time Structure**  
january february, march april (0.92)  
Preserves calendar order through strong sequential structure in the embedding space.

**4. Directional Relationships**  
west east, north northern (0.88)  
Directional concepts cluster closely, including morphological variants.

**5. Evaluative Adjectives**  
new old, good best / own (0.84)  
Captures comparative meaning with some overlap between ranking and possession-like semantics.

**6. Numerical Progression**  
one two, first three (0.78)  
Reflects numerical ordering patterns rather than strict lexical matching.